In [1]:
!pip install -q --force-reinstall "numpy<2"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sagemaker-serve 1.6.0 requires sagemaker-core>=2.6.0, but you have sagemaker-core 1.0.77 which is incompatible.
sagemaker-train 1.6.0 requires sagemaker-core>=2.6.0, but you have sagemaker-core 1.0.77 which is incompatible.


In [2]:
!pip install -q --only-binary=:all: "pyarrow==15.0.2"

In [3]:
!pip install -q "datasets==3.5.0"

In [4]:
!pip install -q \
"transformers==4.52.4" \
"accelerate==1.7.0" \
"peft==0.15.2" \
"trl==0.18.1" \
"sentencepiece==0.1.99" \
einops \
scipy \
safetensors

In [5]:
!pip install -q bitsandbytes

In [6]:
import torch
import transformers
import datasets
import peft
import trl
import bitsandbytes

print("ALL GOOD")

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


ALL GOOD


In [7]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="automotive_en_dataset.jsonl"
)

dataset

DatasetDict({
    train: Dataset({
        features: ['deita_score', 'rw_score', 'instruction', 'conversations'],
        num_rows: 44773
    })
})

In [8]:
small_dataset = dataset["train"].shuffle(seed=42).select(range(4000))

In [9]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

In [10]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token

In [11]:
def format_chat(example):

    human = example["conversations"][0]["value"]
    assistant = example["conversations"][1]["value"]

    messages = [
        {
            "role": "system",
            "content": "You are an automotive expert assistant."
        },
        {
            "role": "user",
            "content": human
        },
        {
            "role": "assistant",
            "content": assistant
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}

In [12]:
small_dataset = small_dataset.map(format_chat)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

In [13]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [14]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [15]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="./qwen3b-automotive",

    num_train_epochs=1,

    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,

    learning_rate=5e-5,

    logging_steps=10,

    save_strategy="no",

    bf16=True,

    optim="paged_adamw_8bit",

    lr_scheduler_type="cosine",

    warmup_ratio=0.03,

    max_seq_length=512,

    packing=False,

    report_to="none"
)

In [16]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=small_dataset,
    args=training_args,
)

Converting train dataset to ChatML:   0%|          | 0/4000 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [17]:
trainer.train()

Step,Training Loss
10,4.150200
20,1.171600
30,0.230300
40,0.000700
50,0.000200
60,0.000100
70,0.000100
80,0.000100
90,0.000100
100,0.000100


TrainOutput(global_step=500, training_loss=0.1110850020024227, metrics={'train_runtime': 420.3155, 'train_samples_per_second': 9.517, 'train_steps_per_second': 1.19, 'total_flos': 1413572493312000.0, 'train_loss': 0.1110850020024227})

In [18]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

Device set to use cuda:0


In [19]:
prompt = "Explain symptoms of a failing alternator."

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|im_end|>")
]

result = pipe(
    text,

    max_new_tokens=120,

    temperature=0.7,
    top_p=0.9,

    repetition_penalty=1.1,

    do_sample=True,

    return_full_text=False,

    eos_token_id=terminators
)

print(result[0]["generated_text"])

A failing or faulty alternator can cause several symptoms that may indicate problems with the electrical system in your vehicle. Here are some common signs to look for:

1. **Dimming Lights**: One of the first and most noticeable signs is dim headlights when you turn on other electrical devices like the radio, windshield wipers, or air conditioning.

2. **Starter Motor Issues**: If the alternator isn't working correctly, it might make the starter motor work harder and longer than normal, leading to engine cranking issues or even stalling during startup.

3. **Battery Problems**: A weak battery


In [21]:
!pip install -q huggingface_hub

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [23]:
from huggingface_hub import login

login()

In [24]:
trainer.model.save_pretrained(
    "./qwen3b-automotive-lora"
)

tokenizer.save_pretrained(
    "./qwen3b-automotive-lora-4k"
)

('./qwen3b-automotive-lora-4k/tokenizer_config.json',
 './qwen3b-automotive-lora-4k/special_tokens_map.json',
 './qwen3b-automotive-lora-4k/chat_template.jinja',
 './qwen3b-automotive-lora-4k/vocab.json',
 './qwen3b-automotive-lora-4k/merges.txt',
 './qwen3b-automotive-lora-4k/added_tokens.json',
 './qwen3b-automotive-lora-4k/tokenizer.json')

In [25]:
trainer.model.push_to_hub(
    "Nasim435/qwen3b-automotive-lora"
)

tokenizer.push_to_hub(
    "Nasim435/qwen3b-automotive-lora"
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/Nasim435/qwen3b-automotive-lora/commit/fd98ab859ae3782d5c2199d826946c733d1f0cf2', commit_message='Upload tokenizer', commit_description='', oid='fd98ab859ae3782d5c2199d826946c733d1f0cf2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Nasim435/qwen3b-automotive-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='Nasim435/qwen3b-automotive-lora'), pr_revision=None, pr_num=None)

In [26]:
merged_model = model.merge_and_unload()

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/peft/tuners/lora/bnb.py:351: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [27]:
merged_model.push_to_hub(
    "Nasim435/Qwen-3B-Automotive-4000"
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/Nasim435/Qwen-3B-Automotive-4000/commit/52ae0c6757a26d983c6f27174c03708388181456', commit_message='Upload Qwen2ForCausalLM', commit_description='', oid='52ae0c6757a26d983c6f27174c03708388181456', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Nasim435/Qwen-3B-Automotive-4000', endpoint='https://huggingface.co', repo_type='model', repo_id='Nasim435/Qwen-3B-Automotive-4000'), pr_revision=None, pr_num=None)